In [1]:
import pandas as pd
from scipy.stats import shapiro
from scipy.stats import kruskal

In [2]:
# Load the dataset from CSV file and define column names
df = pd.read_csv('Biometrics_4HZ/labeled_combined_biometric_data_4HZ.csv', low_memory=False)
df.columns = ['acc_x','acc_y','acc_z','bvp','eda', 'hr', 'SPO2','Pulse_Rate', 'motion','temp','Subject','Session','Label']

In [3]:
# Convert columns to numeric if possible
df[['SPO2', 'Pulse_Rate']] = df[['SPO2', 'Pulse_Rate']].apply(pd.to_numeric, errors='coerce')

In [4]:
# Interpolate NaN values for SPO2 and Pulse_rate subject-session wise
df[['SPO2', 'Pulse_Rate']] = df.groupby(['Subject', 'Session'])[['SPO2', 'Pulse_Rate']].apply(
    lambda group: group.interpolate(method='linear', limit_direction='forward')
).reset_index(drop=True)[['SPO2', 'Pulse_Rate']]

### Check signal normality distribution ###

In [ ]:
# Example: Check normality for SPO2 grouped by Subject and Session
grouped = df.groupby(['Subject', 'Session'])['SPO2']

for (subject, session), data in grouped:
    # Run Shapiro-Wilk test
    stat, p_value = shapiro(data)
    
    # Check if p-value is greater than 0.05 (normality assumption)
    if p_value > 0.05:
        result = "normally distributed"
    else:
        result = "not normally distributed"
    
    # Print the result
    print(f"Subject {subject}, Session {session} - P-value: {p_value:.4f} ({result})")


### Non parametric tests ###

In [6]:
def map_drowsiness_level(x):
    if x < 4:
        return 'Low'
    elif 4 <= x <= 6:
        return 'Med'
    else:
        return 'High'   

df['DrowsinessLevel'] = df['Label'].apply(map_drowsiness_level)

In [7]:
# List of features
features = ['eda', 'temp', 'hr', 'acc_x', 'acc_y', 'acc_z', 'bvp', 'SPO2', 'Pulse_Rate']

In [ ]:
# Function to perform the Kruskal-Wallis test
def perform_kruskal_wallis(df, measurement, level1, level2):
    # Filter data for the two drowsiness levels
    data1 = df[df['DrowsinessLevel'] == level1][measurement].dropna()
    data2 = df[df['DrowsinessLevel'] == level2][measurement].dropna()
    
    # Check if sufficient data is available for the test
    if len(data1) > 0 and len(data2) > 0:
        # Perform the Kruskal-Wallis test
        h_stat, p_value = kruskal(data1, data2)
        test_type = 'Kruskal-Wallis H-test'
    else:
        h_stat, p_value = None, None
        test_type = 'Insufficient Data'
    
    return h_stat, p_value, test_type, len(data1), len(data2)

# Main script to run Kruskal-Wallis tests
kruskal_results = []
drowsiness_levels = ['Low', 'Med', 'High']
from itertools import combinations

# All possible pairs of drowsiness levels
level_pairs = list(combinations(drowsiness_levels, 2))

for measurement in features:
    print(f"\n*** Kruskal-Wallis Test Results for {measurement} ***")
    for level1, level2 in level_pairs:
        h_stat, p_value, test_type, n1, n2 = perform_kruskal_wallis(df, measurement, level1, level2)
        if h_stat is not None:
            print(f"{test_type} between '{level1}' and '{level2}':")
            print(f"H-statistic = {h_stat:.4f}, P-value = {p_value:.4e}, Samples: {n1} (level1), {n2} (level2)")
        else:
            print(f"{test_type} between '{level1}' and '{level2}': Insufficient samples in one or both groups.")
        
        # Store the results
        kruskal_results.append({
            'Measurement': measurement,
            'Level1': level1,
            'Level2': level2,
            'Test_Type': test_type,
            'H_Statistic': h_stat,
            'P_Value': p_value,
            'Samples_Level1': n1,
            'Samples_Level2': n2
        })
